# SVPA Gauge Forecast Regression

Goal: For every flood event where the **Carnation** USGS gauge (`12149000`) crests
**≥ 51 ft** (its flood/yellow stage), regress each SVPA gauge's crest stage against
its USGS *predictor* gauge's crest stage. The two USGS gauges with NWRFC forecasts
— **Falls** (`12144500`, RM 38) and **Carnation** (`12149000`, RM 22) — are the
predictors, so a good fit lets us turn an NWRFC forecast into a forecast for each
SVPA gauge.

Predictor split (by river mile of the SVPA gauge):

| SVPA river mile | Predictor |
|---|---|
| > 29 | Falls (RM 38) |
| ≤ 29 | Carnation (RM 22) |

Each scatter point is one flood event; the fit line is a simple linear regression.
Hover a point to see its **event date** so outliers can be traced back to a specific
flood and checked for bad data.

> SVPA-6 / SVPA-6B are intentionally skipped. A per-gauge bad-crest exclusion layer
> (like `data_files/historic-crest-overrides.json`) is a planned follow-up and is
> **not** applied here yet.

Notebook assumes the working directory is `notebooks/` (paths are relative with `../`),
matching `crest-analysis.ipynb`.

In [57]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn import linear_model

CARNATION_ID = "12149000"   # RM 22, NWRFC forecast
FALLS_ID = "12144500"       # RM 38, NWRFC forecast
EVENT_THRESHOLD_FT = 51.0   # Carnation flood / yellow stage
MATCH_WINDOW = pd.Timedelta(hours=24)
SKIP_GAUGES = {"SVPA-6", "SVPA-6B"}

ROOT = Path("..")

with open(ROOT / "locations.json") as f:
    locations = json.load(f)["locations"]

river_mile = {loc["id"]: loc.get("riverMile") for loc in locations}
loc_name = {loc["id"]: loc["name"] for loc in locations}
with open(ROOT / "data_files" / "forecast-crest-corrections.json") as f:
    _corrections = json.load(f)
# {gauge_id: {excluded event_date, ...}}; keys starting with "__" are docs.
EXCLUSIONS = {
    gid: set(entry.get("exclude", {}))
    for gid, entry in _corrections.items() if not gid.startswith("__")
}

river_mile[FALLS_ID], river_mile[CARNATION_ID]

(38, 22)

In [58]:
def load_crests(source: str, gauge_id: str) -> pd.DataFrame:
    """Load a per-gauge crest CSV (tab-delimited) with utc parsed and sorted."""
    path = ROOT / "crests" / source / gauge_id / f"{source}-{gauge_id}-crests.csv"
    df = pd.read_csv(path, sep="\t")
    df["utc"] = pd.to_datetime(df["utc"])
    return df.sort_values("utc").reset_index(drop=True)

carnation = load_crests("usgs", CARNATION_ID)
falls = load_crests("usgs", FALLS_ID)

events = carnation[carnation["stage"] >= EVENT_THRESHOLD_FT].copy()
print(f"{len(events)} Carnation events >= {EVENT_THRESHOLD_FT} ft "
      f"(of {len(carnation)} detected crests)")
events[["utc", "stage", "discharge"]].head()

105 Carnation events >= 51.0 ft (of 478 detected crests)


,utc,stage,discharge
5,2007-12-04 18:00:00,56.99,32100.0
14,2008-05-18 16:45:00,53.48,18100.0
15,2008-05-21 07:15:00,52.35,15300.0
27,2008-11-08 07:00:00,57.24,34100.0
28,2008-11-13 15:45:00,60.49,63100.0


In [59]:
def match_crest(df: pd.DataFrame, ref_time: pd.Timestamp, direction: str,
                window: pd.Timedelta = MATCH_WINDOW):
    """Return the highest crest in df within `window` of ref_time.

    direction='after'  -> search (ref_time, ref_time + window]   (target downstream
                          of predictor, so it crests later)
    direction='before' -> search [ref_time - window, ref_time)   (target upstream,
                          crests earlier)
    Returns a row (Series) or None.
    """
    if direction == "after":
        mask = (df["utc"] > ref_time) & (df["utc"] <= ref_time + window)
    elif direction == "before":
        mask = (df["utc"] >= ref_time - window) & (df["utc"] < ref_time)
    else:
        raise ValueError(direction)
    window_df = df[mask]
    if window_df.empty:
        return None
    return window_df.loc[window_df["stage"].idxmax()]


def build_pairs(target_id: str, target_correction=None, apply_exclusions=True):
    """Pair predictor stage with target SVPA stage for each qualifying event.

    Returns a DataFrame with columns: event_date, predictor_stage, target_stage.
    """
    target_rm = river_mile[target_id]
    if target_rm is not None and target_rm > 29:
        predictor_id, predictor_rm, predictor_df = FALLS_ID, river_mile[FALLS_ID], falls
    else:
        predictor_id, predictor_rm, predictor_df = CARNATION_ID, river_mile[CARNATION_ID], carnation

    target_df = load_crests("svpa", target_id)
    if target_correction is not None:
        target_df = target_correction(target_df)
    # Predictor upstream of target -> target crests later -> look AFTER predictor crest.
    direction = "after" if predictor_rm > target_rm else "before"

    rows = []
    for _, ev in events.iterrows():
        # Predictor crest for this event.
        if predictor_id == CARNATION_ID:
            t_pred, s_pred = ev["utc"], ev["stage"]
        else:
            # Falls is upstream of Carnation, so its crest precedes the event.
            pred_crest = match_crest(predictor_df, ev["utc"], "before")
            if pred_crest is None:
                continue
            t_pred, s_pred = pred_crest["utc"], pred_crest["stage"]

        target_crest = match_crest(target_df, t_pred, direction)
        if target_crest is None:
            continue
        rows.append({
            "event_date": ev["utc"].date().isoformat(),
            "predictor_stage": float(s_pred),
            "target_stage": float(target_crest["stage"]),
        })

    pairs = pd.DataFrame(rows)
    if apply_exclusions and not pairs.empty:
        excluded = EXCLUSIONS.get(target_id, set())
        pairs = pairs[~pairs["event_date"].isin(excluded)].reset_index(drop=True)
    return pairs, predictor_id, direction

pairs, pid, d = build_pairs("SVPA-25")
print(f"SVPA-25 predictor={pid} direction={d} -> {len(pairs)} paired events")
pairs.head()

SVPA-25 predictor=12149000 direction=before -> 26 paired events


,event_date,predictor_stage,target_stage
0,2021-01-03,52.55,67.26
1,2021-01-14,55.33,69.02
2,2021-02-23,53.03,67.73
3,2021-10-29,55.26,69.43
4,2021-11-13,57.12,71.57


In [60]:
def build_model(pairs: pd.DataFrame):
    X = pairs["predictor_stage"].to_numpy().reshape(-1, 1)
    y = pairs["target_stage"].to_numpy()
    regr = linear_model.LinearRegression().fit(X, y)
    return regr, regr.score(X, y)


def plot_model(regr, pairs, predictor_id, target_id, score):
    Xv = pairs["predictor_stage"].to_numpy()
    yv = pairs["target_stage"].to_numpy()
    xfit = np.linspace(Xv.min() - 1, Xv.max() + 1, 200)
    yfit = regr.predict(xfit[:, None])

    pred_label = f"{loc_name.get(predictor_id, predictor_id)} ({predictor_id})"
    targ_label = f"{loc_name.get(target_id, target_id)} ({target_id})"
    title = (f"{pred_label} → {targ_label}<br>"
             f"<sub>n={len(pairs)}  R²={score:.3f}  "
             f"y = {regr.coef_[0]:.3f}x + {regr.intercept_:.2f}</sub>")

    fig = go.Figure()
    fig.add_scatter(
        x=Xv, y=yv, mode="markers", name="events",
        text=pairs["event_date"],
        hovertemplate=(f"{pred_label}: %{{x:.2f}} ft<br>"
                       f"{targ_label}: %{{y:.2f}} ft<br>%{{text}}<extra></extra>"),
    )
    fig.add_scatter(x=xfit, y=yfit, mode="lines", name="fit", hoverinfo="skip")
    fig.update_layout(title=title, xaxis_title=f"{pred_label} stage (ft)",
                      yaxis_title=f"{targ_label} stage (ft)", height=500)
    fig.show()

regr, score = build_model(pairs)
plot_model(regr, pairs, pid, "SVPA-25", score)

In [61]:
svpa_gauges = [loc["id"] for loc in locations
               if loc["source"] == "SVPA" and loc["id"] not in SKIP_GAUGES]

summary = []
for gid in svpa_gauges:
    pairs, predictor_id, direction = build_pairs(gid)
    if len(pairs) < 2:
        print(f"{gid}: only {len(pairs)} paired events — skipping plot")
        summary.append({"gauge": gid, "predictor": predictor_id,
                        "n": len(pairs), "r2": np.nan})
        continue
    regr, score = build_model(pairs)
    summary.append({"gauge": gid, "predictor": predictor_id,
                    "n": len(pairs), "r2": round(score, 3)})
    plot_model(regr, pairs, predictor_id, gid, score)

pd.DataFrame(summary)

,gauge,predictor,n,r2
0,SVPA-37,12144500,11,0.787
1,SVPA-36,12144500,22,0.921
2,SVPA-34,12144500,23,0.960
3,SVPA-33,12144500,33,0.835
4,SVPA-29,12149000,15,0.690
5,SVPA-26,12149000,26,0.989
6,SVPA-25,12149000,26,0.964
7,SVPA-19,12149000,4,0.985
8,SVPA-17,12149000,13,0.713
9,SVPA-15,12149000,26,0.926


## Probability of reaching redStage vs predictor stage

For each SVPA gauge we already have a strong linear fit `target ≈ slope·predictor + intercept`.
Treating the fit residuals as Gaussian, the probability that the gauge reaches its
**redStage** for a given predictor stage *x* is

$$P(\text{target} \ge \text{redStage} \mid x) = \Phi\!\left(\frac{\hat{y}(x) - \text{redStage}}{\sigma}\right)$$

where $\hat{y}(x)$ is the regression prediction and $\sigma$ is the residual standard
error of the fit ($\sqrt{\text{SSE}/(n-2)}$). `redStage` values come from the Floodzilla
`APIGetLocationInfo` endpoint. The dots show the actual events (at y=1 if they reached
redStage, y=0 otherwise) so you can sanity-check the curve and still spot outliers.

In [62]:
import requests
from scipy.stats import norm

_info = requests.get(
    "https://floodzilla.com/api/client/APIGetLocationInfo?regionId=1", timeout=30
).json()
red_stage = {g["id"]: g.get("redStage") for g in _info if g.get("redStage") is not None}
{k: red_stage[k] for k in svpa_gauges if k in red_stage}

{'SVPA-37': 88.6,
 'SVPA-36': 74.12,
 'SVPA-34': 75.85,
 'SVPA-33': 81.95,
 'SVPA-29': 75.67,
 'SVPA-26': 66.51,
 'SVPA-25': 68.43,
 'SVPA-19': 43.94,
 'SVPA-17': 44.49,
 'SVPA-15': 42.91,
 'SVPA-15B': 45.25,
 'SVPA-12': 36.22}

In [63]:
def residual_sigma(regr, pairs):
    """Residual standard error of the linear fit: sqrt(SSE / (n - 2))."""
    X = pairs["predictor_stage"].to_numpy().reshape(-1, 1)
    y = pairs["target_stage"].to_numpy()
    resid = y - regr.predict(X)
    n = len(y)
    return float(np.sqrt(np.sum(resid ** 2) / (n - 2))) if n > 2 else np.nan


def plot_probability_curve(regr, pairs, predictor_id, target_id, red):
    sigma = residual_sigma(regr, pairs)
    if not np.isfinite(sigma) or sigma == 0:
        print(f"{target_id}: cannot estimate residual sigma (n={len(pairs)}) — skipping")
        return None

    Xv = pairs["predictor_stage"].to_numpy()
    xline = np.linspace(Xv.min() - 1, Xv.max() + 3, 300)
    yhat = regr.predict(xline[:, None])
    prob = norm.cdf((yhat - red) / sigma)

    # Predictor stage giving each probability of reaching redStage.
    # P = 0.5 when yhat == red  ->  x = (red - intercept) / slope.
    slope, intercept = regr.coef_[0], regr.intercept_
    def x_for_prob(p):
        return (red + norm.ppf(p) * sigma - intercept) / slope
    x50, x90 = x_for_prob(0.5), x_for_prob(0.9)

    pred_label = f"{loc_name.get(predictor_id, predictor_id)} ({predictor_id})"
    targ_label = f"{loc_name.get(target_id, target_id)} ({target_id})"
    hit = (pairs["target_stage"] >= red).astype(int)

    fig = go.Figure()
    fig.add_scatter(x=xline, y=prob, mode="lines", name="P(reach redStage)",
                    hovertemplate=f"{pred_label}: %{{x:.2f}} ft<br>"
                                  "P=%{y:.0%}<extra></extra>")
    fig.add_scatter(x=Xv, y=hit, mode="markers", name="events",
                    marker=dict(color=hit, colorscale=[[0, "#1f77b4"], [1, "#d62728"]],
                                symbol="circle-open", size=8),
                    text=pairs["event_date"],
                    hovertemplate=(f"{pred_label}: %{{x:.2f}} ft<br>"
                                   f"{targ_label} reached red: %{{y}}<br>"
                                   "%{text}<extra></extra>"))
    for p, xv, dash in [(0.5, x50, "dash"), (0.9, x90, "dot")]:
        fig.add_vline(x=xv, line=dict(color="gray", dash=dash))
    fig.update_layout(
        title=(f"P({targ_label} ≥ redStage {red} ft) vs {pred_label}<br>"
               f"<sub>n={len(pairs)}  σ={sigma:.2f} ft  "
               f"P50 @ {x50:.1f} ft  P90 @ {x90:.1f} ft</sub>"),
        xaxis_title=f"{pred_label} stage (ft)",
        yaxis_title="probability of reaching redStage",
        yaxis=dict(range=[-0.05, 1.05], tickformat=".0%"), height=500)
    fig.show()
    return {"gauge": target_id, "predictor": predictor_id, "redStage": red,
            "sigma": round(sigma, 2), "pred_stage_P50": round(x50, 1),
            "pred_stage_P90": round(x90, 1)}

In [64]:
prob_summary = []
for gid in svpa_gauges:
    if gid not in red_stage:
        print(f"{gid}: no redStage from API — skipping")
        continue
    pairs, predictor_id, direction = build_pairs(gid)
    if len(pairs) < 3:
        print(f"{gid}: only {len(pairs)} paired events — skipping")
        continue
    regr, _ = build_model(pairs)
    row = plot_probability_curve(regr, pairs, predictor_id, gid, red_stage[gid])
    if row:
        prob_summary.append(row)

pd.DataFrame(prob_summary)

,gauge,predictor,redStage,sigma,pred_stage_P50,pred_stage_P90
0,SVPA-37,12144500,88.60,1.19,12.7,14.2
1,SVPA-36,12144500,74.12,0.63,10.4,11.1
2,SVPA-34,12144500,75.85,0.44,12.2,12.8
3,SVPA-33,12144500,81.95,1.18,14.2,15.6
4,SVPA-29,12149000,75.67,1.39,55.5,58.2
5,SVPA-26,12149000,66.51,0.26,50.8,51.2
6,SVPA-25,12149000,68.43,0.52,53.7,54.4
7,SVPA-19,12149000,43.94,0.37,52.0,52.6
8,SVPA-17,12149000,44.49,1.28,51.8,52.9
9,SVPA-15,12149000,42.91,0.89,53.6,54.6


## Corrected charts — SVPA-37 (Sno Falls Golf)

SVPA-37 has a known reading error starting **November 2022**. Checked against the
predictor (Falls), every post-Nov-2022 crest sits ~2.6 ft **below** the historic
Falls→SVPA-37 relationship, so **2.65 ft is added** to each SVPA-37 crest dated
**2022-11-01 onward**. This lifts the corrected fit from R² ≈ 0.33 (raw) to ≈ 0.99.
The main charts above are left uncorrected on purpose so the raw error stays visible.

This correction is applied *only* to SVPA-37 via the `target_correction` hook on
`build_pairs`; no other gauge is affected.

In [65]:
def correct_svpa37(df):
    """Add 2.65 ft to SVPA-37 crests dated 2022-11-01 onward.

    Post-Nov-2022 readings sit ~2.6 ft below the historic Falls->SVPA-37
    relationship; adding 2.65 ft restores the fit (R^2 0.33 -> 0.99).
    """
    df = df.copy()
    bad = df["utc"] >= pd.Timestamp("2022-11-01")
    df.loc[bad, "stage"] = df.loc[bad, "stage"] + 2.65
    return df


# Original (uncorrected) fit, for comparison.
orig_pairs, _, _ = build_pairs("SVPA-37")
orig_regr, orig_score = build_model(orig_pairs)

# Corrected fit.
pairs37, pid37, _ = build_pairs("SVPA-37", target_correction=correct_svpa37)
regr37, score37 = build_model(pairs37)

print(f"SVPA-37 uncorrected:  n={len(orig_pairs)}  R²={orig_score:.3f}")
print(f"SVPA-37 corrected:    n={len(pairs37)}  R²={score37:.3f}")

SVPA-37 uncorrected:  n=11  R²=0.787
SVPA-37 corrected:    n=11  R²=0.988


In [66]:
plot_model(regr37, pairs37, pid37, "SVPA-37", score37)
plot_probability_curve(regr37, pairs37, pid37, "SVPA-37", red_stage["SVPA-37"])

{'gauge': 'SVPA-37',
 'predictor': '12144500',
 'redStage': 88.6,
 'sigma': 0.32,
 'pred_stage_P50': np.float64(12.5),
 'pred_stage_P90': np.float64(12.9)}

## Crest exclusions & outlier candidates

Bad crests are suppressed per gauge via `data_files/forecast-crest-corrections.json`,
keyed by gauge id → Carnation `event_date` (the date shown in the plot hover). Excluded
events are dropped from that gauge's regression and probability curve automatically by
`build_pairs` (no other gauge affected). The SVPA-37 datum offset is handled separately
by `correct_svpa37`, not here.

The report below ranks the remaining points by how far they sit from each gauge's fit
(standardized residual *z*). Use it to find candidates: confirm the reading really is
bad, then add it to the JSON, e.g.

```json
"SVPA-29": { "name": "Jubilee Farm", "exclude": { "2020-02-08": "stuck sensor" } }
```

Re-run the notebook and the point drops out; the next-worst candidate surfaces.

In [67]:
from IPython.display import HTML

FLOODZILLA_BASE = "http://localhost:8081"


def _gage_url(gauge_id, event_date, pad_days=1):
    d = pd.Timestamp(event_date)
    frm = (d - pd.Timedelta(days=pad_days)).date()
    to = (d + pd.Timedelta(days=pad_days)).date()
    return f"{FLOODZILLA_BASE}/gage/{gauge_id}?from={frm}&to={to}"


def exclusion_candidates(n_top=2, z_flag=2.0):
    """Largest standardized residuals per gauge (after current exclusions)."""
    rows = []
    for gid in svpa_gauges:
        pairs, predictor_id, _ = build_pairs(gid)
        if len(pairs) < 4:
            continue
        regr, _ = build_model(pairs)
        X = pairs["predictor_stage"].to_numpy().reshape(-1, 1)
        resid = pairs["target_stage"].to_numpy() - regr.predict(X)
        sigma = resid.std(ddof=1)
        z = resid / sigma if sigma else np.zeros_like(resid)
        for k in np.argsort(-np.abs(z))[:n_top]:
            ev = pairs.iloc[k]["event_date"]
            rows.append({
                "gauge": gid, "predictor": predictor_id, "event_date": ev,
                "predictor_stage": round(float(pairs.iloc[k]["predictor_stage"]), 2),
                "target_stage": round(float(pairs.iloc[k]["target_stage"]), 2),
                "residual_ft": round(float(resid[k]), 2),
                "z": round(float(z[k]), 2),
                "flag": "***" if abs(z[k]) >= z_flag else "",
                "url": _gage_url(gid, ev),
            })
    out = pd.DataFrame(rows)
    return out.reindex(out["z"].abs().sort_values(ascending=False).index)


def show_candidates(**kwargs):
    """Render exclusion_candidates() with the event_date as a Floodzilla link."""
    df = exclusion_candidates(**kwargs).copy()
    df["event_date"] = [
        f'<a href="{u}" target="_blank">{d}</a>'
        for d, u in zip(df["event_date"], df["url"])
    ]
    return HTML(df.drop(columns="url").to_html(escape=False, index=False))

show_candidates()

gauge,predictor,event_date,predictor_stage,target_stage,residual_ft,z,flag
SVPA-26,12149000,2022-03-02,58.73,74.93,0.72,2.85,***
SVPA-25,12149000,2025-12-11,60.76,76.95,1.44,2.81,***
SVPA-15,12149000,2022-11-05,56.90,44.31,-2.39,-2.73,***
SVPA-12,12149000,2022-10-31,51.50,36.50,-2.54,-2.62,***
SVPA-33,12144500,2025-12-11,19.95,85.22,-3.01,-2.60,***
SVPA-34,12144500,2026-03-19,14.58,79.13,1.11,2.58,***
SVPA-36,12144500,2022-03-02,16.91,80.77,-1.32,-2.14,***
SVPA-29,12149000,2022-03-02,58.73,74.99,-2.78,-2.07,***
SVPA-25,12149000,2021-01-14,55.33,69.02,-1.05,-2.04,***
SVPA-15B,12149000,2025-03-25,53.59,45.51,0.95,1.98,
